# 06 — Exporter les objets OSM d’Auvergne-Rhône-Alpes en tableau

Ce notebook produit un tableau CSV avec une ligne par objet géographique OSM retenu dans la région AURA et sa marge technique d’1 km. Chaque ligne conserve l’identifiant OSM, le nom, la grande famille, la catégorie précise, les coordonnées d’affichage et les tags OSM disponibles.

Le tableau est pratique à filtrer dans VS Code, LibreOffice ou Excel. Les objets restent des objets OSM : un objet n’est pas toujours un établissement. Les coordonnées servent à l’affichage et ne désignent pas forcément l’entrée d’un lieu.

## 1. Préparer le projet

Choisis le noyau **Python (IGN / OSM)** déjà utilisé par le projet. La première exécution lance l’extraction régionale si le catalogue n’existe pas encore ; les suivantes réutilisent le résultat lorsque les sources et les règles n’ont pas changé.

In [1]:
from pathlib import Path
import json
import sys
import pandas as pd
from IPython.display import display

current_directory = Path.cwd().resolve()
project_directory = next(
    (directory for directory in [current_directory, *current_directory.parents]
     if (directory / "regional-data" / "AGENTS.md").exists()),
    None,
)
if project_directory is None:
    raise FileNotFoundError("Open this notebook from the repository directory.")

module_directory = project_directory / "regional-data"
output_directory = module_directory / "data" / "processed"
sys.path.insert(0, str(module_directory))
import geopandas as gpd
from places_catalog import CATEGORY_LABELS

print(f"Project: {project_directory}")
print(f"Output: {output_directory}")

Project: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz
Output: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/processed


## 2. Charger les objets OSM déjà extraits

Le notebook 02 a déjà assemblé et filtré les objets des deux fichiers OSM pour AURA. Nous lisons son GeoPackage, qui conserve les géométries originales et évite de retraiter les PBF. Le périmètre est la région et sa marge technique d’1 km.

La colonne `inside_region` identifie les objets qui touchent le contour régional. `countable=False` marque des doublons probables conservés pour audit.


In [2]:
objects_path = output_directory / "osm_objects.gpkg"
if not objects_path.exists():
    raise FileNotFoundError(
        "Run notebooks 01 and 02 first to prepare the regional OSM objects."
    )

places = gpd.read_file(objects_path)
print(f"Regional objects loaded: {len(places):,}")
print(f"Objects inside AURA: {int(places['inside_region'].sum()):,}")
print(f"Potential duplicates retained for audit: {int((~places['countable']).sum()):,}")


Regional objects loaded: 192,803
Objects inside AURA: 191,837
Potential duplicates retained for audit: 70


## 3. Construire le tableau CSV simple

Le tableau ci-dessous contient les champs les plus pratiques pour démarrer. `broad_category` donne la grande famille et `point_category` la catégorie OSM précise, par exemple `health` et `amenity=pharmacy`. `category_matches_json` conserve les autres catégories si un objet correspond à plusieurs familles.

`tags_json` contient les attributs d’origine présents dans OSM, comme l’adresse, les horaires, l’accès ou l’opérateur. Une information absente reste inconnue.

In [3]:
from shapely.geometry import Point
from pyproj import Transformer

family_by_tag = {
    "amenity": {"food_and_drink": {"restaurant", "cafe", "bar", "pub", "fast_food", "food_court", "ice_cream", "biergarten"},
                "education": {"school", "kindergarten", "college", "university", "language_school", "music_school", "driving_school", "research_institute", "training"},
                "health": {"hospital", "clinic", "doctors", "dentist", "pharmacy", "nursing_home", "social_facility", "baby_hatch"},
                "mobility": {"parking", "parking_entrance", "parking_space", "bicycle_parking", "bicycle_rental", "bicycle_repair_station", "charging_station", "fuel", "bus_station", "car_sharing", "car_rental", "taxi", "motorcycle_parking", "ferry_terminal", "vehicle_inspection"},
                "public_services": {"townhall", "courthouse", "prison", "post_office", "post_box", "recycling", "waste_disposal", "waste_transfer_station", "toilets", "drinking_water", "water_point", "shelter", "public_bath", "shower", "childcare", "animal_shelter"},
                "finance": {"bank", "atm", "bureau_de_change", "payment_terminal"},
                "culture_and_community": {"library", "cinema", "theatre", "arts_centre", "community_centre", "conference_centre", "events_venue", "place_of_worship", "public_bookcase", "studio"},
                "emergency": {"police", "fire_station", "rescue_station", "ranger_station"}},
    "healthcare": {"health"}, "shop": {"retail"}, "office": {"professional_services"},
    "craft": {"craft"}, "tourism": {"tourism"}, "leisure": {"sports_and_leisure"},
    "sport": {"sports_and_leisure"}, "historic": {"heritage"},
    "public_transport": {"mobility"}, "railway": {"mobility"}, "highway": {"mobility"},
    "aeroway": {"mobility"}, "aerialway": {"mobility"}, "emergency": {"emergency"},
    "natural": {"nature"}, "waterway": {"water"}, "water": {"water"},
    "man_made": {"infrastructure"}, "place": {"settlement"}, "mountain_pass": {"nature"},
}
family_labels = {**CATEGORY_LABELS}
for key in sorted(family_labels):
    pass

metric_region = gpd.read_file(output_directory / "aura_region.geojson").to_crs(2154).geometry.iloc[0]
metric_points = places.to_crs(2154).geometry.representative_point()
coordinates = gpd.GeoSeries(metric_points, crs="EPSG:2154").to_crs(4326)

rows = []
for position, row in enumerate(places.itertuples()):
    tags = {
        "amenity": row.amenity,
        "tourism": row.tourism,
    }
    try:
        tags.update(json.loads(row.tags_json))
    except (AttributeError, TypeError, json.JSONDecodeError):
        pass
    matches = []
    for tag_key, tag_value in tags.items():
        if not tag_value or tag_value in {"no", "disused", "abandoned", "razed", "demolished", "construction", "proposed"}:
            continue
        family = None
        if tag_key == "amenity":
            family = next((name for name, values in family_by_tag["amenity"].items() if tag_value in values), "other_amenities")
        else:
            families = family_by_tag.get(tag_key)
            if families:
                family = next(iter(families)) if isinstance(families, set) else next(iter(families))
                if isinstance(families, set):
                    family = next(iter(families))
        if family:
            matches.append({"broad_category": family, "point_category": f"{tag_key}={tag_value}"})
    primary = matches[0] if matches else {"broad_category": "other_amenities", "point_category": "other"}
    rows.append({
        "osm_uid": row.osm_uid,
        "osm_type": row.osm_type,
        "osm_id": str(row.osm_id),
        "name": row.name,
        "broad_category": primary["broad_category"],
        "broad_category_label_fr": family_labels.get(primary["broad_category"], "Autres lieux"),
        "point_category": primary["point_category"],
        "category_matches_json": json.dumps(matches, ensure_ascii=False),
        "geometry_type": row.geometry.geom_type,
        "longitude": float(coordinates.iloc[position].x),
        "latitude": float(coordinates.iloc[position].y),
        "inside_region": bool(row.inside_region),
        "countable": bool(row.countable),
        "duplicate_of": row.duplicate_of,
        "access": row.access,
        "operator": row.operator,
        "source_files": row.source_files,
        "tags_json": json.dumps(tags, ensure_ascii=False, sort_keys=True),
    })

objects_for_table = pd.DataFrame(rows).sort_values(
    ["broad_category", "point_category", "name", "osm_uid"], kind="stable"
)
csv_path = output_directory / "osm_aura_table.csv"
objects_for_table.to_csv(csv_path, index=False, encoding="utf-8")
print(f"CSV ready: {csv_path}")
print(f"Rows: {len(objects_for_table):,} | Columns: {len(objects_for_table.columns)}")
display(objects_for_table.head(20))

CSV ready: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/processed/osm_aura_table.csv
Rows: 192,803 | Columns: 18


,osm_uid,osm_type,osm_id,name,broad_category,broad_category_label_fr,point_category,category_matches_json,geometry_type,longitude,latitude,inside_region,countable,duplicate_of,access,operator,source_files,tags_json
6176,node/10937970197,node,10937970197,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,3.324993,45.408266,True,True,,,,auvergne-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
34132,node/1827775609,node,1827775609,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,4.836955,45.069962,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
42367,node/281732188,node,281732188,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,6.078831,46.140605,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
43476,node/302637232,node,302637232,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,5.924427,45.565604,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
58255,node/5846740867,node,5846740867,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,4.816649,46.178333,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
73509,node/8529285868,node,8529285868,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",Point,6.707105,45.939511,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
103323,way/128100936,way,128100936,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",MultiPolygon,4.404853,44.386630,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
166018,way/568733429,way,568733429,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",MultiPolygon,4.236495,44.393217,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
168727,way/62649015,way,62649015,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",MultiPolygon,6.107377,45.906498,True,True,,,,rhone-alpes-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"
172549,way/67552307,way,67552307,,culture_and_community,Culture et vie collective,amenity=arts_centre,"[{""broad_category"": ""culture_and_community"", ""...",MultiPolygon,3.092404,45.771578,True,True,,,,auvergne-261016.osm.pbf,"{""amenity"": ""arts_centre"", ""tourism"": """"}"


## 4. Filtrer par grande famille ou catégorie

Modifie `selected_category` pour obtenir un sous-tableau d’une famille. Pour ne rien filtrer, laisse la valeur `None`. Le fichier régional complet reste disponible ; le sous-tableau sert seulement à explorer ou à exporter une sélection.

In [4]:
selected_category = None  # Example: "health", "retail", or "mobility"

if selected_category is None:
    filtered_table = objects_for_table
else:
    if selected_category not in category_labels:
        raise ValueError(f"Unknown broad category: {selected_category}")
    filtered_table = objects_for_table.loc[
        objects_for_table["broad_category"] == selected_category
    ].copy()

print(f"Selected rows: {len(filtered_table):,}")
display(filtered_table[[
    "osm_uid", "name", "broad_category_label_fr", "point_category",
    "geometry_type", "longitude", "latitude", "inside_region",
]].head(50))

Selected rows: 192,803


,osm_uid,name,broad_category_label_fr,point_category,geometry_type,longitude,latitude,inside_region
6176,node/10937970197,,Culture et vie collective,amenity=arts_centre,Point,3.324993,45.408266,True
34132,node/1827775609,,Culture et vie collective,amenity=arts_centre,Point,4.836955,45.069962,True
42367,node/281732188,,Culture et vie collective,amenity=arts_centre,Point,6.078831,46.140605,True
43476,node/302637232,,Culture et vie collective,amenity=arts_centre,Point,5.924427,45.565604,True
58255,node/5846740867,,Culture et vie collective,amenity=arts_centre,Point,4.816649,46.178333,True
73509,node/8529285868,,Culture et vie collective,amenity=arts_centre,Point,6.707105,45.939511,True
103323,way/128100936,,Culture et vie collective,amenity=arts_centre,MultiPolygon,4.404853,44.386630,True
166018,way/568733429,,Culture et vie collective,amenity=arts_centre,MultiPolygon,4.236495,44.393217,True
168727,way/62649015,,Culture et vie collective,amenity=arts_centre,MultiPolygon,6.107377,45.906498,True
172549,way/67552307,,Culture et vie collective,amenity=arts_centre,MultiPolygon,3.092404,45.771578,True


## 5. Résumer les volumes par catégories

Ces tableaux montrent ce que les fichiers OSM recensent dans les données utilisées. Un petit nombre peut refléter un manque de cartographie, pas nécessairement une absence sur le terrain.

In [5]:
family_summary = (
    objects_for_table.groupby(["broad_category", "broad_category_label_fr"], dropna=False)
    .size().rename("object_count").sort_values(ascending=False).reset_index()
)
category_summary = (
    objects_for_table.groupby(
        ["broad_category_label_fr", "point_category"], dropna=False
    ).size().rename("object_count").sort_values(ascending=False).reset_index()
)
display(family_summary)
display(category_summary.head(50))

,broad_category,broad_category_label_fr,object_count
0,mobility,Mobilité et stationnement,114431
1,food_and_drink,Restauration et boissons,22939
2,culture_and_community,Culture et vie collective,13379
3,tourism,Tourisme et hébergement,11250
4,education,Éducation,10617
5,health,Santé,7409
6,public_services,Services publics et collectifs,6707
7,finance,Banques et services financiers,3626
8,emergency,Secours et sécurité,1893
9,other_amenities,Autres équipements et services,552


,broad_category_label_fr,point_category,object_count
0,Mobilité et stationnement,amenity=parking,87338
1,Mobilité et stationnement,amenity=bicycle_parking,24124
2,Restauration et boissons,amenity=restaurant,13286
3,Éducation,amenity=school,9137
4,Culture et vie collective,amenity=place_of_worship,7598
5,Tourisme et hébergement,tourism=viewpoint,5024
6,Services publics et collectifs,amenity=townhall,4266
7,Restauration et boissons,amenity=fast_food,4003
8,Tourisme et hébergement,tourism=artwork,3843
9,Culture et vie collective,amenity=community_centre,3224


## 6. Exporter une sélection facultative

Si tu as choisi une `selected_category`, cette cellule exporte aussi la sélection sous forme d’un second CSV. Le fichier complet AURA n’est jamais remplacé.

In [6]:
if selected_category is not None:
    selected_path = output_directory / f"osm_aura_{selected_category}_table.csv"
    filtered_table.to_csv(selected_path, index=False, encoding="utf-8")
    print(f"Filtered CSV ready: {selected_path}")

print("Complete regional table:", csv_path)
print("Geometry-preserving file:", output_directory / "osm_places_catalog.gpkg")

Complete regional table: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/processed/osm_aura_table.csv
Geometry-preserving file: /home/doryan-epitech/Desktop/Hackaton/ign-osm/Datz_Data-analyze-for-Citiz/regional-data/data/processed/osm_places_catalog.gpkg


## À retenir

- Le fichier principal est `data/processed/osm_aura_table.csv`.
- Il contient une ligne par objet OSM retenu et les catégories de lieux définies par ce projet.
- Le GeoPackage `osm_places_catalog.gpkg` garde les formes géographiques originales ; le CSV est plus simple pour le tri et les filtres.
- `countable=False` indique un doublon probable point/surface conservé pour audit.
- Les décomptes ne constituent pas un registre exhaustif des lieux présents sur le terrain.